<a href="https://colab.research.google.com/github/rinyan-thai23/--chiangra-thai-news/blob/main/DataForSEO_Sandbox_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 DataForSEO Sandbox エンドポイント・シート風ビジュアルビューア

このノートブックは、**DataForSEOの各エンドポイントを完全無料（Sandbox環境）で呼び出し、返ってきたネストしたJSONをスプレッドシートのような表形式（Pandas対話型テーブル）で即座にプレビュー・検証するための特化ツール**です。

### 💡 このツールの特徴
1. **完全無料（残高消費ゼロ）**: 全リクエストが Sandbox 宛てに送信される安全設計。
2. **ドロップダウン選択**: カテゴリ ➔ エンドポイントを選ぶだけで、最小限の必要パラメータが自動セット。
3. **JSONの自動シート化**: 深い階層のJSON（`tasks[0].result[0].items` など）を自動で平坦化（flatten）し、スプレッドシート風の表として展開。
4. **対話型操作**: Colabの `data_table` により、検索、並び替え、ページ送り、全画面表示が自由自在。

---
## 1. 環境初期化 & 認証設定
必要なライブラリのセットアップと、Sandbox通信用の認証情報を設定します。

In [ ]:
#@title 🔑 DataForSEO 認証情報設定 { run: "auto" }
!pip install pandas requests ipywidgets -q

import base64
import json
import requests
import pandas as pd
from google.colab import data_table
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Colabのインタラクティブテーブル表示を有効化（スプレッドシート風のUI）
data_table.enable_dataframe_formatter()

# API認証情報
API_LOGIN = "mjflash@rinyan.net" #@param {type:"string"}
API_PASSWORD = "471366f800950797" #@param {type:"string"}

# 常にSandbox環境へ接続（完全無料・残高消費なし）
SANDBOX_URL = "https://sandbox.dataforseo.com/v3"

def get_headers():
    if not API_LOGIN or not API_PASSWORD:
        return None
    credentials = f"{API_LOGIN}:{API_PASSWORD}"
    encoded = base64.b64encode(credentials.encode()).decode("utf-8")
    return {
        "Authorization": f"Basic {encoded}",
        "Content-Type": "application/json"
    }

print("✅ 準備完了: インタラクティブテーブル機能を有効化しました。" if get_headers() else "⚠️ API_LOGIN と API_PASSWORD を入力してセルを実行してください。")

---
## 2. エンドポイント定義マスター（サンプル呼び出しテンプレート付き）
各エンドポイントを叩くための「デフォルトリクエスト（最小限の引数）」をプリセットしたマスター辞書です。

In [ ]:
# 各エンドポイントのパスとデフォルトリクエストパラメータ定義
ENDPOINT_CONFIGS = {
    # --- SERP API ---
    "SERP: Google 自然検索 (organic/live)": {
        "path": "/serp/google/organic/live",
        "payload": [{"keyword": "SEO ツール", "location_code": 2392, "language_code": "ja", "depth": 20}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "Google検索結果（オーガニック順位・URL・スニペットなど）"
    },
    "SERP: Google マップ (maps/live)": {
        "path": "/serp/google/maps/live",
        "payload": [{"keyword": "カフェ", "location_code": 2392, "language_code": "ja", "depth": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "Googleマップのローカル検索結果（MEO順位・店舗情報など）"
    },
    "SERP: YouTube 検索 (organic/live)": {
        "path": "/serp/youtube/organic/live",
        "payload": [{"keyword": "python 入門", "depth": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "YouTube動画の検索結果（タイトル・再生回数・チャンネル名等）"
    },
    "SERP: Google ニュース (news/live)": {
        "path": "/serp/google/news/live",
        "payload": [{"keyword": "AI 技術", "location_code": 2392, "language_code": "ja", "depth": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "Googleニュース枠の最新記事一覧"
    },

    # --- Keywords Data API ---
    "Keywords Data: Google Ads 検索ボリューム (search_volume/live)": {
        "path": "/keywords_data/google_ads/search_volume/live",
        "payload": [{"keywords": ["seo 対策", "web マーケティング", "python 自動化"], "location_code": 2392, "language_code": "ja"}],
        "items_path": ["tasks", 0, "result"],
        "desc": "Google Ads公式の月間検索ボリューム・クリック単価(CPC)・競合性"
    },
    "Keywords Data: キーワード拡張 (keywords_for_keywords/live)": {
        "path": "/keywords_data/google_ads/keywords_for_keywords/live",
        "payload": [{"keywords": ["データ分析"], "location_code": 2392, "language_code": "ja"}],
        "items_path": ["tasks", 0, "result"],
        "desc": "指定キーワードから派生する関連キーワード候補群"
    },
    "Keywords Data: サイト関連キーワード抽出 (keywords_for_site/live)": {
        "path": "/keywords_data/google_ads/keywords_for_site/live",
        "payload": [{"target": "example.com", "location_code": 2392, "language_code": "ja"}],
        "items_path": ["tasks", 0, "result"],
        "desc": "WebサイトURLを分析して自動抽出した関連キーワードリスト"
    },
    "Keywords Data: Google Trends トレンド推移 (explore/live)": {
        "path": "/keywords_data/google_trends/explore/live",
        "payload": [{"keywords": ["AI", "DX"], "location_code": 2392, "type": "web"}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "指定KWのGoogleトレンド時系列スコア推移"
    },

    # --- DataForSEO Labs API ---
    "Labs: 競合ドメイン自動抽出 (competitors_domain/live)": {
        "path": "/dataforseo_labs/google/competitors_domain/live",
        "payload": [{"target": "example.com", "location_code": 2392, "language_code": "ja", "limit": 5}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "検索結果でキーワードが重複している真の競合ドメイン"
    },
    "Labs: ドメインの獲得キーワード一覧 (ranked_keywords/live)": {
        "path": "/dataforseo_labs/google/ranked_keywords/live",
        "payload": [{"target": "example.com", "location_code": 2392, "language_code": "ja", "limit": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "指定サイトがGoogleで順位を獲得している全キーワード一覧"
    },
    "Labs: 競合とのキーワード重複・ギャップ (domain_intersection/live)": {
        "path": "/dataforseo_labs/google/domain_intersection/live",
        "payload": [{"target1": "example.com", "target2": "youtube.com", "location_code": 2392, "language_code": "ja", "limit": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "自社と競合のランクインキーワードの重複・抜け漏れギャップ"
    },
    "Labs: サジェストキーワード抽出 (keyword_suggestions/live)": {
        "path": "/dataforseo_labs/google/keyword_suggestions/live",
        "payload": [{"keyword": "マーケティング", "location_code": 2392, "language_code": "ja", "limit": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "サジェスト・ロングテールキーワードと各検索ボリューム"
    },
    "Labs: キーワード難易度一括判定 (bulk_keyword_difficulty/live)": {
        "path": "/dataforseo_labs/google/bulk_keyword_difficulty/live",
        "payload": [{"keywords": ["seo 初心者", "python 自動化 ツール", "データ分析 副業"], "location_code": 2392, "language_code": "ja"}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "キーワードのSEO難易度スコア(KD 0〜100)"
    },
    "Labs: 検索意図一括判定 (search_intent/live)": {
        "path": "/dataforseo_labs/google/search_intent/live",
        "payload": [{"keywords": ["おすすめ ノートパソコン", "python インストール方法", "ログイン"], "language_code": "ja"}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "キーワードの検索意図（情報/ナビ/比較検討/購入）"
    },
    "Labs: ドメインの総合SEO概要 (domain_rank_overview/live)": {
        "path": "/dataforseo_labs/google/domain_rank_overview/live",
        "payload": [{"target": "example.com", "location_code": 2392, "language_code": "ja"}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "ドメイン全体の推定オーガニック流入数(ETV)や順位分布"
    },

    # --- Business Data API ---
    "Business Data: Googleビジネスプロフィール詳細 (my_business_info/live)": {
        "path": "/business_data/google/my_business_info/live",
        "payload": [{"keyword": "スターバックス 渋谷", "location_code": 2392, "language_code": "ja"}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "店舗の営業時間・電話番号・星評価・緯度経度・カテゴリ"
    },

    # --- Merchant API ---
    "Merchant: Google ショッピング商品一覧 (products/live)": {
        "path": "/merchant/google/products/live",
        "payload": [{"keyword": "ワイヤレスイヤホン", "location_code": 2392, "language_code": "ja", "depth": 10}],
        "items_path": ["tasks", 0, "result", 0, "items"],
        "desc": "Googleショッピングの出品商品一覧（商品名・価格・店舗名・評価）"
    }
}

print(f"マスター登録エンドポイント数: {len(ENDPOINT_CONFIGS)} 件")

---
## 3. 【メインUI】エンドポイント選択 ➔ Sandbox実行 ➔ シート風自動展開
プルダウンからエンドポイントを選び、「実行」ボタンを押すだけで、
**複雑なJSONを自動分解してスプレッドシート風の対話型テーブル（検索・ソート・CSV保存対応）** として画面に表示します。

In [ ]:
#@title 🚀 Sandbox データ取得 & シート風プレビューUI

# UI要素の構築
endpoint_select = widgets.Dropdown(
    options=list(ENDPOINT_CONFIGS.keys()),
    value=list(ENDPOINT_CONFIGS.keys())[0],
    description="API選択:",
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'}
)

run_btn = widgets.Button(
    description="Sandboxでデータ取得（完全無料）",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width='280px')
)

info_output = widgets.Output()
table_output = widgets.Output()

def extract_items_by_path(data, path_list):
    """指定された階層パスを辿ってデータリストを安全に取り出す"""
    curr = data
    for p in path_list:
        if isinstance(curr, dict) and p in curr:
            curr = curr[p]
        elif isinstance(curr, list) and isinstance(p, int) and len(curr) > p:
            curr = curr[p]
        else:
            return None
    return curr

def on_endpoint_change(change):
    info_output.clear_output()
    cfg = ENDPOINT_CONFIGS[change['new']]
    with info_output:
        display(HTML(f"""
        <div style="background: #f1f5f9; padding: 10px 14px; border-left: 4px solid #3b82f6; border-radius: 4px; margin: 6px 0;">
            <b>🎯 機能概要:</b> {cfg['desc']}<br>
            <b>📍 エンドポイント:</b> <code>{cfg['path']}</code>
        </div>
        """))

def on_run_clicked(b):
    table_output.clear_output()
    headers = get_headers()
    if not headers:
        with table_output:
            print("❌ 認証情報が未入力です。「1. 環境初期化 & 認証設定」でログインIDとパスワードを入力してください。")
        return

    selected_name = endpoint_select.value
    cfg = ENDPOINT_CONFIGS[selected_name]
    url = f"{SANDBOX_URL}{cfg['path']}"

    with table_output:
        print(f"⏳ Sandbox サーバーにリクエスト中... ({cfg['path']})")
        try:
            res = requests.post(url, headers=headers, json=cfg['payload'], timeout=30)
            res_json = res.json()
        except Exception as e:
            print(f"通信エラーが発生しました: {e}")
            return

        # レスポンスチェック
        if res_json.get("status_code") != 20000:
            print("⚠️ APIエラー:", res_json.get("status_message"))
            print(json.dumps(res_json, indent=2))
            return

        # itemsの抽出
        items = extract_items_by_path(res_json, cfg["items_path"])

        if items is None or len(items) == 0:
            # フォールバック: result配下全体を平坦化
            tasks = res_json.get("tasks", [])
            if tasks and "result" in tasks[0]:
                items = tasks[0]["result"]

        if not items:
            print("取得結果: データが空または抽出パスが一致しませんでした。生のレスポンス:")
            print(json.dumps(res_json, indent=2))
            return

        # ⭐️ 核心部分: ネストしたJSONを自動平坦化してスプレッドシート化
        df = pd.json_normalize(items)

        # 表示を整理（多すぎる空列や複雑な内部オブジェクト列は文字列表現に）
        for col in df.columns:
            if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                df[col] = df[col].astype(str)

        clear_output()
        display(HTML(f"""
        <div style="margin: 10px 0; color: #166534; font-weight: bold;">
            ✅ 取得成功（Sandbox環境 / 課金ゼロ）: 全 <b>{len(df)}</b> 行 × <b>{len(df.columns)}</b> 列のデータをシート化しました。
        </div>
        """))

        # インタラクティブテーブルとして描画（フィルタ・ソート・ページ送り付き）
        display(data_table.DataTable(df, include_index=False, num_rows_per_page=10))

# イベント登録
endpoint_select.observe(on_endpoint_change, names='value')
run_btn.on_click(on_run_clicked)

# 初期表示
on_endpoint_change({'new': endpoint_select.value})
display(widgets.VBox([
    endpoint_select,
    info_output,
    run_btn,
    widgets.HTML("<hr style='margin: 15px 0;'>"),
    table_output
]))


---
## 4. このシート風ビューアの活用方法 & 次のステップ

1. **各エンドポイントの「カラム名（キー名）」の確認**
   - 表のヘッダー（例: `avg_position`, `intersections`, `keyword_data.keyword` など）を見ることで、GASや自作ツールでどのカラムを抜き出せばいいかが一目瞭然になります。
2. **新しいエンドポイントの追加方法**
   - セクション2の `ENDPOINT_CONFIGS` 辞書に、新しいパスとサンプル引数を追加するだけで、自動的にこのセレクターの選択肢に反映されます。
3. **エージェントへの引き継ぎ方**
   - このノートブックをAIエージェント（CursorやClaude Codeなど）に渡し、「この `ENDPOINT_CONFIGS` に記載されているAPIを使って、指定したドメインを分析するGASスクリプトを書いて」と指示すれば、データ構造のズレがない完璧なコードが生成されます。